In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

# Define Indian stocks
stocks = {
    "Reliance":   "RELIANCE.NS",
    "TCS":        "TCS.NS",
    "Infosys":    "INFY.NS",
    "HDFC Bank":  "HDFCBANK.NS",
    "Wipro":      "WIPRO.NS",
    "ICICI Bank": "ICICIBANK.NS",
    "SBI":        "SBIN.NS",
    "HCL Tech":   "HCLTECH.NS"
}

all_data = []

for name, ticker in stocks.items():
    print(f"Downloading {name}...")

    df = yf.download(ticker,
                     start="2020-01-01",
                     end="2024-01-01")

    df.columns       = [col[0] for col in df.columns]
    df['Stock_Name'] = name
    df['Ticker']     = ticker

    all_data.append(df)
    print(f"✅ {name} — {len(df)} rows downloaded")

# Combine all stocks
data = pd.concat(all_data)
data = data.reset_index()

print(f"\nTotal rows : {len(data)}")
print(f"Total stocks: {data['Stock_Name'].nunique()}")
print("\nRows per stock:")
print(data['Stock_Name'].value_counts())

[*********************100%***********************]  1 of 1 completed


✅ Reliance — 992 rows downloaded


[*********************100%***********************]  1 of 1 completed


✅ TCS — 992 rows downloaded


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


✅ Infosys — 992 rows downloaded
✅ HDFC Bank — 992 rows downloaded


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


✅ Wipro — 992 rows downloaded
✅ ICICI Bank — 992 rows downloaded


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

✅ SBI — 992 rows downloaded
✅ HCL Tech — 992 rows downloaded

Total rows : 7936
Total stocks: 8

Rows per stock:
Stock_Name
Reliance      992
TCS           992
Infosys       992
HDFC Bank     992
Wipro         992
ICICI Bank    992
SBI           992
HCL Tech      992
Name: count, dtype: int64


In [2]:
data

,Date,Close,High,Low,Open,Volume,Stock_Name,Ticker
0,2020-01-01,675.324158,683.152852,673.490062,679.081936,14004468,Reliance,RELIANCE.NS
1,2020-01-02,686.821228,689.348791,676.397899,676.397899,17710316,Reliance,RELIANCE.NS
2,2020-01-03,687.648865,689.661956,681.318790,685.792313,20984698,Reliance,RELIANCE.NS
3,2020-01-06,671.700806,683.510891,670.135054,679.976842,24519177,Reliance,RELIANCE.NS
4,2020-01-07,682.034546,686.463335,677.068889,679.529321,16683622,Reliance,RELIANCE.NS
...,...,...,...,...,...,...,...,...
7931,2023-12-22,1325.239868,1329.452897,1289.270797,1296.518982,2303438,HCL Tech,HCLTECH.NS
7932,2023-12-26,1321.117432,1329.452799,1306.621064,1325.239770,1363265,HCL Tech,HCLTECH.NS
7933,2023-12-27,1333.711060,1337.289806,1320.573727,1327.731220,1402329,HCL Tech,HCLTECH.NS
7934,2023-12-28,1334.073486,1343.360222,1330.449394,1332.397366,1928616,HCL Tech,HCLTECH.NS


In [3]:
df['Daily_Return'] = df['Close'].pct_change()
#Today's price - Yesterday's price / Yesterday's price
#Example:
#Yesterday: ₹100
#Today:     ₹105
#Return = (105-100)/100 = 0.05 = 5% gain

In [4]:
df['SMA_10'] = df['Close'].rolling(10).mean()
df['SMA_20'] = df['Close'].rolling(20).mean()
df['SMA_50'] = df['Close'].rolling(50).mean()

In [5]:
#SMA = Simple Moving Average
#Average of last 10/20/50 days closing price

#Example SMA_10:
#Day 10 = average of days 1-10
#Day 11 = average of days 2-11
#Day 12 = average of days 3-12

#Why useful?
#SMA_10 > SMA_50 → short term trend is UP
#SMA_10 < SMA_50 → short term trend is DOWN

In [6]:
df['Volatility'] = df['Daily_Return'].rolling(10).std()

In [7]:
#Standard deviation of last 10 days returns
#High volatility = price jumping up and down a lot = risky
#Low volatility  = price stable = safe

In [8]:
df['EMA_10'] = df['Close'].ewm(span=10).mean()

#EMA = Exponential Moving Average
#Similar to SMA but:
#Recent days have MORE weight
#Old days have LESS weight

#Example:
#Yesterday's price matters more than 10 days ago
#More responsive to recent changes than SMA

In [9]:
df['Momentum'] = df['Close'] - df['Close'].shift(10)

In [10]:
#Today's price - Price 10 days ago

#Positive momentum = price went UP in 10 days
#Negative momentum = price went DOWN in 10 days

#Example:
#Today:        ₹1000
#10 days ago:  ₹900
#Momentum = ₹100 (positive = upward trend)

In [11]:
# RSI Calculation

delta    = df['Close'].diff()          # daily price change
gain     = delta.where(delta > 0, 0)  # only positive changes
loss     = -delta.where(delta < 0, 0) # only negative changes
avg_gain = gain.rolling(14).mean()    # average gain over 14 days
avg_loss = loss.rolling(14).mean()    # average loss over 14 days
rs       = avg_gain / avg_loss        # ratio of gain to loss
df['RSI'] = 100 - (100 / (1 + rs))   # convert to 0-100 scale

In [12]:
#RSI = Relative Strength Index
#Tells you if stock is overbought or oversold

#RSI > 70 → Overbought → price likely to DROP
#RSI < 30 → Oversold   → price likely to RISE
#RSI 30-70 → Neutral

#Example:
#RSI = 80 → Stock went up too much → expect drop
#RSI = 25 → Stock fell too much    → expect rise

In [13]:
df['Target'] = df['Close'].shift(-1)

#shift(-1) moves price UP by one row
#So today's Target = tomorrow's Close price

#This is what model tries to PREDICT!

#Example:
#Row 0: Close=₹675, Target=₹686  ← tomorrow's price
#Row 1: Close=₹686, Target=₹687
#Row 2: Close=₹687, Target=₹671

In [14]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 992 entries, 2020-01-01 to 2023-12-29
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Close         992 non-null    float64
 1   High          992 non-null    float64
 2   Low           992 non-null    float64
 3   Open          992 non-null    float64
 4   Volume        992 non-null    int64  
 5   Stock_Name    992 non-null    str    
 6   Ticker        992 non-null    str    
 7   Daily_Return  991 non-null    float64
 8   SMA_10        983 non-null    float64
 9   SMA_20        973 non-null    float64
 10  SMA_50        943 non-null    float64
 11  Volatility    982 non-null    float64
 12  EMA_10        992 non-null    float64
 13  Momentum      982 non-null    float64
 14  RSI           979 non-null    float64
 15  Target        991 non-null    float64
dtypes: float64(13), int64(1), str(2)
memory usage: 131.8 KB


In [18]:
def add_features(df):
    df['Daily_Return'] = df['Close'].pct_change()
    df['SMA_10']       = df['Close'].rolling(10).mean()
    df['SMA_20']       = df['Close'].rolling(20).mean()
    df['SMA_50']       = df['Close'].rolling(50).mean()
    df['Volatility']   = df['Daily_Return'].rolling(10).std()
    df['EMA_10']       = df['Close'].ewm(span=10).mean()
    df['Momentum']     = df['Close'] - df['Close'].shift(10)

    delta    = df['Close'].diff()
    gain     = delta.where(delta > 0, 0)
    loss     = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs       = avg_gain / avg_loss
    df['RSI'] = 100 - (100 / (1 + rs))

    df['Target'] = df['Close'].shift(-1)
    return df

In [19]:
data = df.groupby('Stock_Name',group_keys=False).apply(add_features)

#Without groupby:
#Last row of Reliance + First row of TCS
#= Wrong SMA calculation! ❌

#With groupby:
#Reliance features calculated separately ✅
#TCS features calculated separately      ✅
#INFY features calculated separately     ✅

#Each stock's indicators are independent

In [20]:
data = data.dropna().reset_index(drop=True)

In [21]:
data

,Close,High,Low,Open,Volume,Ticker,Daily_Return,SMA_10,SMA_20,SMA_50,Volatility,EMA_10,Momentum,RSI,Target
0,390.060242,417.297635,384.525790,411.921320,5321388,HCLTECH.NS,-0.080943,434.727228,456.281932,463.143330,0.038718,433.758091,-71.354889,22.889483,389.704346
1,389.704346,408.956331,313.052226,363.692435,6859998,HCLTECH.NS,-0.000912,428.390271,451.490672,461.923531,0.038994,425.748031,-63.369568,21.976149,356.339661
2,356.339661,379.347149,353.295707,370.808290,5281571,HCLTECH.NS,-0.085615,421.776599,445.096410,460.011225,0.042039,413.127956,-66.136719,19.233214,356.300049
3,356.300049,378.398331,350.528417,355.786140,5148153,HCLTECH.NS,-0.000111,413.945346,438.336469,457.916885,0.039541,402.795361,-78.312531,20.328682,343.333618
4,343.333618,367.487534,339.222302,359.739312,4936397,HCLTECH.NS,-0.036392,403.603830,431.252362,455.611286,0.036157,391.983922,-103.415161,18.537994,326.888428
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
937,1288.410034,1311.060609,1286.643223,1287.594591,2841508,HCLTECH.NS,-0.012431,1290.575354,1246.859778,1187.578401,0.025283,1290.789818,85.211426,63.948392,1325.239868
938,1325.239868,1329.452897,1289.270797,1296.518982,2303438,HCLTECH.NS,0.028585,1299.508752,1252.916541,1191.435295,0.025410,1297.053464,89.333984,68.061402,1321.117432
939,1321.117432,1329.452799,1306.621064,1325.239770,1363265,HCLTECH.NS,-0.003111,1307.015161,1259.666406,1195.500515,0.025616,1301.428731,75.064087,71.335348,1333.711060
940,1333.711060,1337.289806,1320.573727,1327.731220,1402329,HCLTECH.NS,0.009533,1315.780933,1266.706201,1200.209792,0.025539,1307.298245,87.657715,71.499701,1334.073486


In [22]:
print(f"Final shape: {data.shape}")
print(f"Total stocks: {data['Ticker'].nunique()}")
print(f"Stocks: {data['Ticker'].unique()}")
print()
print(data[['Ticker','Close','SMA_10','RSI','Target']].head(10))
print()
print("Any NaN values?")
print(data.isnull().sum())

Final shape: (942, 15)
Total stocks: 1
Stocks: <StringArray>
['HCLTECH.NS']
Length: 1, dtype: str

       Ticker       Close      SMA_10        RSI      Target
0  HCLTECH.NS  390.060242  434.727228  22.889483  389.704346
1  HCLTECH.NS  389.704346  428.390271  21.976149  356.339661
2  HCLTECH.NS  356.339661  421.776599  19.233214  356.300049
3  HCLTECH.NS  356.300049  413.945346  20.328682  343.333618
4  HCLTECH.NS  343.333618  403.603830  18.537994  326.888428
5  HCLTECH.NS  326.888428  391.771964  17.770573  351.753876
6  HCLTECH.NS  351.753876  381.375092  31.391675  329.813812
7  HCLTECH.NS  329.813812  369.590659  23.778422  349.500610
8  HCLTECH.NS  349.500610  361.810800  26.553561  361.795013
9  HCLTECH.NS  361.795013  355.548965  30.880750  353.967651

Any NaN values?
Close           0
High            0
Low             0
Open            0
Volume          0
Ticker          0
Daily_Return    0
SMA_10          0
SMA_20          0
SMA_50          0
Volatility      0
EMA_10         

In [24]:
# Features to train on
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import joblib
features = ['SMA_10', 'SMA_20', 'SMA_50',
            'Daily_Return', 'Volatility',
            'EMA_10', 'Momentum', 'RSI', 'Volume']

X = data[features]  # input features
y = data['Target']  # what to predict

# Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows : {len(X_test)}")

Training rows: 753
Testing rows : 189


In [25]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import joblib

# Train with 50 trees instead of 100
model = RandomForestRegressor(
    n_estimators=50,
    random_state=42)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
r2 = r2_score(y_test, predictions)

print(f"R2 Score: {r2:.4f}")

# Save smaller model
joblib.dump(model, 'indian_stock_model.pkl',
            compress=3)  # compress=3 reduces size!

import os
size = os.path.getsize('indian_stock_model.pkl')
print(f"Model size: {size/1024/1024:.1f} MB")

R2 Score: 0.9944
Model size: 0.8 MB
